# Model Final: E6 ResNet50, Deteksi Pneumonia dari Chest X-Ray

## 0. Setup dan parameter

In [ ]:
import os, re, sys, json, glob, time, hashlib
from datetime import date
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, tensorflow as tf, keras
from keras import layers
from sklearn import metrics as skm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight

print('python', sys.version.split()[0], '| tensorflow', tf.__version__, '| keras', keras.__version__,
      '| scikit-learn', sklearn.__version__, '| backend', keras.backend.backend())

EXP = 'final_E6_resnet50'
CONFIG = dict(
    seed=42, img=224, batch=32, val_folds=7,                              # data, split per pasien (val sekitar 1/7)
    backbone='resnet50', head_units=256, dropout=0.3, l2=1e-4,            # ResNet50 (ImageNet) + head
    rotation=0.04, zoom=0.15, translation=0.1, contrast=0.1, flip=True,   # augmentasi, aktif hanya saat training
    class_weight='balanced',                                              # penanganan imbalance
    lr_a=1e-3, epochs_a=10, patience_a=3,                                 # Tahap A: backbone beku
    lr_b=1e-4, epochs_b=30, patience_b=5, unfreeze_n=50, cosine_alpha=0.01,  # Tahap B: fine-tuning
    target_recall=0.95, n_boot=1000,                                      # evaluasi: threshold dari val, CI bootstrap
)
SPLIT_MD5_EXPECTED = '18f95e62'                   # split yang sama dengan semua eksperimen E1–E6
SEED, IMG, BATCH = CONFIG['seed'], CONFIG['img'], CONFIG['batch']

keras.utils.set_random_seed(SEED)
if tf.config.list_physical_devices('GPU'):
    keras.mixed_precision.set_global_policy('mixed_float16')
print('GPU:', tf.config.list_physical_devices('GPU'), '| policy:', keras.mixed_precision.global_policy().name)

WORK = Path('/kaggle/working/outputs')
OUT = WORK / EXP
(OUT / 'figs').mkdir(parents=True, exist_ok=True)
json.dump(CONFIG, open(OUT / 'config.json', 'w'), indent=2)
print(EXP, CONFIG)

## 1. Inventaris data

In [ ]:
roots = [p for p in glob.glob('/kaggle/input/**/chest_xray', recursive=True)
         if '__MACOSX' not in p and os.path.isdir(f'{p}/train')]
assert roots, 'Dataset belum ditambahkan: Add Input -> chest-xray-pneumonia'
ROOT = min(roots, key=len)                        # dataset punya salinan ter-nest, ambil yang teratas
print('ROOT =', ROOT)

df = pd.DataFrame([dict(path=f'{o}/{c}/{f}', label=int(c == 'PNEUMONIA'), origin=o)
                   for o in ['train', 'val', 'test'] for c in ['NORMAL', 'PNEUMONIA']
                   for f in sorted(os.listdir(f'{ROOT}/{o}/{c}'))
                   if not f.startswith('.') and f.lower().endswith(('.jpeg', '.jpg', '.png'))])
print(df.groupby(['origin', 'label']).size().unstack().rename(columns={0: 'NORMAL', 1: 'PNEUMONIA'}))
assert df.origin.value_counts().to_dict() == {'train': 5216, 'test': 624, 'val': 16}, 'Jumlah file beda dari A1: STOP'

## 2. Split per pasien → `split.csv`

In [ ]:
def patient_id(path, label):
    m = re.search(r'person([0-9]+)_' if label == 1 else r'IM-([0-9]+)-', os.path.basename(path))
    return f'{label}_{m.group(1)}' if m else None

df['group'] = [patient_id(p, y) for p, y in zip(df.path, df.label)]
print('Gagal parse ID pasien:', df.group.isna().sum())
df['group'] = df.group.fillna(df.label.astype(str) + '_' + df.path.map(os.path.basename))  # gagal parse: 1 file = 1 grup

# A3 gagal: ID yang sama muncul di train/val bawaan dan test. Buang gambar train/val bawaan dengan ID itu; test tidak diubah.
overlap = set(df[df.origin != 'test'].group) & set(df[df.origin == 'test'].group)
drop = (df.origin != 'test') & df.group.isin(overlap)
print(f'ID beririsan dengan test: {len(overlap)} | gambar train/val bawaan dibuang: {drop.sum()} dari {(df.origin != "test").sum()}')
df = df[~drop].reset_index(drop=True)

trval = df[df.origin != 'test'].reset_index(drop=True)
sgkf = StratifiedGroupKFold(n_splits=CONFIG['val_folds'], shuffle=True, random_state=SEED)
_, va_idx = next(sgkf.split(trval, trval.label, trval.group))
trval['split'] = 'train'
trval.loc[va_idx, 'split'] = 'val'
split = pd.concat([trval, df[df.origin == 'test'].assign(split='test')], ignore_index=True)[['path', 'label', 'group', 'split']]
split.to_csv(WORK / 'split.csv', index=False)     # path relatif terhadap ROOT
SPLIT_MD5 = hashlib.md5((WORK / 'split.csv').read_bytes()).hexdigest()[:8]

g = {s: set(split.group[split.split == s]) for s in ['train', 'val', 'test']}
assert not (g['train'] & g['val'] or g['train'] & g['test'] or g['val'] & g['test']), 'Ada pasien di dua split'
print(split.groupby(['split', 'label']).size().unstack().rename(columns={0: 'NORMAL', 1: 'PNEUMONIA'}))
print('split.csv md5:', SPLIT_MD5)
assert SPLIT_MD5 == SPLIT_MD5_EXPECTED, 'Split beda dari eksperimen E1–E6: STOP'

## 3. Pipeline tf.data

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load(path, label):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)         # grayscale -> RGB
    img = tf.image.resize(img, (IMG, IMG), antialias=True)
    img = tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8)   # cache sebagai uint8
    return img, tf.reshape(tf.cast(label, tf.float32), [1])             # label (1,) -> batch (B, 1)

def make_ds(d, train):
    ds = tf.data.Dataset.from_tensor_slices(((ROOT + '/' + d.path).values, d.label.values))
    ds = ds.map(load, num_parallel_calls=AUTOTUNE).cache()
    if train:
        ds = ds.shuffle(2048, seed=SEED)
    ds = ds.batch(BATCH).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_df, val_df, test_df = (split[split.split == s].reset_index(drop=True) for s in ['train', 'val', 'test'])
train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df, False), make_ds(test_df, False)

## 4. Model ResNet50

In [ ]:
def build_model(cfg):
    base = keras.applications.ResNet50(include_top=False, weights='imagenet', input_shape=(IMG, IMG, 3))
    base.trainable = False
    aug = [layers.RandomRotation(cfg['rotation']), layers.RandomZoom(cfg['zoom']),
           layers.RandomTranslation(cfg['translation'], cfg['translation']), layers.RandomContrast(cfg['contrast'])]
    if cfg['flip']:
        aug.append(layers.RandomFlip('horizontal'))
    inputs = keras.Input((IMG, IMG, 3))                    # float32, nilai 0-255
    x = keras.Sequential(aug, name='augment')(inputs)      # aktif hanya saat training
    x = keras.Sequential([layers.Lambda(keras.applications.resnet50.preprocess_input)],
                         name='preprocess')(x)             # mode 'caffe': RGB -> BGR lalu kurangi mean ImageNet
    x = base(x, training=False)                            # BatchNorm tetap mode inference
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dense(cfg['head_units'], activation='relu', name='fc', kernel_regularizer=keras.regularizers.l2(cfg['l2']))(x)
    x = layers.Dropout(cfg['dropout'], name='drop')(x)
    out = layers.Dense(1, activation='sigmoid', dtype='float32', name='out')(x)
    return keras.Model(inputs, out), base

def unfreeze_top(base, n):
    base.trainable = True
    for l in base.layers[:-n]:
        l.trainable = False
    for l in base.layers:
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False

model, base = build_model(CONFIG)
x = np.random.default_rng(SEED).uniform(0, 255, (2, IMG, IMG, 3)).astype('float32')
assert np.allclose(model.get_layer('preprocess')(x), keras.applications.resnet50.preprocess_input(x.copy()),
                   atol=0.5), 'Preprocessing ResNet salah'   # skala ±150, toleransi untuk float16
print('Uji preprocessing lolos')
model.summary()

## 5. Training dua tahap
Tahap A: backbone beku, latih head. Tahap B: buka `unfreeze_n` layer teratas (BatchNorm tetap beku), CosineDecay.

In [ ]:
cw = compute_class_weight(CONFIG['class_weight'], classes=np.array([0, 1]), y=train_df.label.values)
class_weight = {0: cw[0], 1: cw[1]}

def metrics():
    return ['accuracy', keras.metrics.AUC(name='auc'), keras.metrics.Recall(name='recall'), keras.metrics.Precision(name='precision')]

def callbacks(patience):
    return [keras.callbacks.EarlyStopping('val_loss', patience=patience, restore_best_weights=True),
            keras.callbacks.ModelCheckpoint(OUT / 'best.keras', monitor='val_loss', save_best_only=True),
            keras.callbacks.CSVLogger(OUT / 'history.csv', append=True)]

t0 = time.time()
model.compile(keras.optimizers.Adam(CONFIG['lr_a']), 'binary_crossentropy', metrics=metrics())
model.fit(train_ds, validation_data=val_ds, epochs=CONFIG['epochs_a'], class_weight=class_weight,
          callbacks=callbacks(CONFIG['patience_a']))

unfreeze_top(base, CONFIG['unfreeze_n'])
sched = keras.optimizers.schedules.CosineDecay(CONFIG['lr_b'], decay_steps=CONFIG['epochs_b'] * len(train_ds),
                                               alpha=CONFIG['cosine_alpha'])
model.compile(keras.optimizers.Adam(sched), 'binary_crossentropy', metrics=metrics())   # wajib compile ulang
model.fit(train_ds, validation_data=val_ds, epochs=CONFIG['epochs_b'], class_weight=class_weight,
          callbacks=callbacks(CONFIG['patience_b']))
TRAIN_MIN = (time.time() - t0) / 60
print(f'Waktu training: {TRAIN_MIN:.1f} menit')

## 6. Evaluasi
Threshold dipilih dari **val** (target recall 0,95), dikunci, lalu test diprediksi **sekali**.

In [ ]:
def pick_threshold(y, p, target_recall):
    fpr, tpr, thr = skm.roc_curve(y, p)
    return float(thr[np.where(tpr >= target_recall)[0][0]])   # titik pertama yang mencapai target = specificity tertinggi

def predict(d, ds):
    return d.assign(y_true=d.label, y_prob=model.predict(ds, verbose=0).ravel())[['path', 'group', 'y_true', 'y_prob']]

def rec_spec_auc(y, p, yhat):
    return dict(recall=skm.recall_score(y, yhat), specificity=skm.recall_score(y, yhat, pos_label=0),
                auc=skm.roc_auc_score(y, p))

def fmt(m, k):
    return f"{m[k]:.3f} [{m['ci95'][k][0]:.3f}–{m['ci95'][k][1]:.3f}]"

val_preds = predict(val_df, val_ds)
THR = pick_threshold(val_preds.y_true, val_preds.y_prob, CONFIG['target_recall'])
test_preds = predict(test_df, test_ds)
test_preds['y_pred'] = (test_preds.y_prob >= THR).astype(int)
val_preds.to_csv(OUT / 'val_preds.csv', index=False)
test_preds.to_csv(OUT / 'test_preds.csv', index=False)

y, p, yhat = test_preds.y_true.values, test_preds.y_prob.values, test_preds.y_pred.values
m = dict(threshold=THR, **rec_spec_auc(y, p, yhat), precision=skm.precision_score(y, yhat, zero_division=0),
         f1=skm.f1_score(y, yhat), accuracy=skm.accuracy_score(y, yhat), pr_auc=skm.average_precision_score(y, p),
         train_minutes=round(TRAIN_MIN, 1))
rng = np.random.default_rng(SEED)
boot = pd.DataFrame([rec_spec_auc(y[i], p[i], yhat[i])
                     for i in (rng.integers(0, len(y), len(y)) for _ in range(CONFIG['n_boot']))])
m['ci95'] = {k: boot[k].quantile([0.025, 0.975]).tolist() for k in boot}
json.dump(m, open(OUT / 'metrics.json', 'w'), indent=2)
print(json.dumps(m, indent=2))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
skm.ConfusionMatrixDisplay.from_predictions(y, yhat, display_labels=['NORMAL', 'PNEUMONIA'], ax=ax[0], colorbar=False)
skm.RocCurveDisplay.from_predictions(y, p, ax=ax[1])
skm.PrecisionRecallDisplay.from_predictions(y, p, ax=ax[2])
ax[0].set_title(f'Confusion matrix (threshold {THR:.3f})'); ax[1].set_title('ROC'); ax[2].set_title('Precision-Recall')
fig.savefig(OUT / 'figs/eval.png', dpi=120, bbox_inches='tight'); plt.show()

print('Tempel ke tabel Hasil di EXPERIMENTS.md:')
print(f"| {EXP} | {date.today()} | backbone=resnet50 (final) | {fmt(m, 'recall')} | {fmt(m, 'specificity')} | {fmt(m, 'auc')} "
      f"| {THR:.3f} | {SPLIT_MD5} | {TRAIN_MIN:.0f} | v? |")

## 7. Grad-CAM (8 contoh TP, TN, FP, FN dari test)
Angka di judul tiap gambar = **confidence score** `p(pneumonia)` dari model; diprediksi PNEUMONIA bila `p ≥ threshold`.
Grad-CAM dihitung dari logit (skor sebelum sigmoid), sehingga peta tidak kosong untuk skor yang sangat dekat 0 atau 1.

In [ ]:
def gradcam(img):                                          # img: (224, 224, 3) float 0-255
    x = model.get_layer('preprocess')(img[None])
    fmap = tf.cast(base(x, training=False), tf.float32)    # ekstraktor fitur = output backbone
    out = model.get_layer('out')
    with tf.GradientTape() as tape:
        tape.watch(fmap)
        h = fmap
        for name in ['gap', 'fc', 'drop']:                 # head
            h = model.get_layer(name)(h)
        logit = tf.matmul(tf.cast(h, tf.float32), out.kernel) + out.bias   # skor sebelum sigmoid, seperti paper Grad-CAM
    w = tf.reduce_mean(tape.gradient(logit, fmap), axis=(1, 2))
    cam = tf.nn.relu(tf.reduce_sum(fmap * w[:, None, None, :], axis=-1))[0]
    cam = tf.image.resize((cam / (tf.reduce_max(cam) + 1e-8))[..., None], (IMG, IMG))[..., 0]
    return cam.numpy(), float(tf.sigmoid(logit)[0, 0])

case = np.select([(y == 1) & (yhat == 1), (y == 0) & (yhat == 0), (y == 0) & (yhat == 1)], ['TP', 'TN', 'FP'], 'FN')
fig, ax = plt.subplots(4, 8, figsize=(24, 13))
for r, c in enumerate(['TP', 'TN', 'FP', 'FN']):
    idx = np.where(case == c)[0]
    if len(idx) == 0:
        ax[r][0].set_title(f'{c}: tidak ada contoh', fontsize=10)
    for a, i in zip(ax[r], rng.choice(idx, min(8, len(idx)), replace=False)):
        img = load(ROOT + '/' + test_df.path[i], 0)[0].numpy().astype('float32')
        cam, _ = gradcam(img)
        a.imshow(img.astype('uint8')); a.imshow(cam, cmap='jet', alpha=0.4)
        a.set_title(f'{c} | p(pneumonia)={p[i]:.2f}', fontsize=10)
    for a in ax[r]:
        a.axis('off')
fig.suptitle(f'{EXP}: Grad-CAM pada test, threshold {THR:.3f}', fontsize=14)
fig.savefig(OUT / 'figs/gradcam.png', dpi=100, bbox_inches='tight'); plt.show()

## 8. Kurva training dan val vs test

In [ ]:
h = pd.read_csv(OUT / 'history.csv')
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, k in zip(ax, ['loss', 'auc']):
    h[[k, f'val_{k}']].plot(ax=a, title=k, marker='o')
    for b in h.index[h.epoch == 0][1:]:
        a.axvline(b - 0.5, ls='--', c='gray')          # garis putus = mulai Tahap B
fig.savefig(OUT / 'figs/history.png', dpi=120, bbox_inches='tight'); plt.show()

vy, vp = val_preds.y_true.values, val_preds.y_prob.values
display(pd.DataFrame({'val': rec_spec_auc(vy, vp, (vp >= THR).astype(int)), 'test': rec_spec_auc(y, p, yhat)}).round(3))

## 9. Muat ulang model
Cara memakai `best.keras` di notebook lain. `custom_objects` wajib, karena layer preprocessing ResNet50 memakai `Lambda`;
tanpa itu `load_model` gagal. Threshold ada di `metrics.json`.

In [ ]:
final_model = keras.models.load_model(OUT / 'best.keras',
                                      custom_objects={'preprocess_input': keras.applications.resnet50.preprocess_input})
assert np.allclose(final_model.predict(test_ds, verbose=0).ravel(), p, atol=1e-3), 'best.keras beda dari model yang dievaluasi'
print('best.keras bisa dimuat ulang; prediksinya sama dengan hasil evaluasi')